# Практическое задание: Featuretools, DFS

In [1]:
import warnings
import numpy as np
import pandas as pd
import featuretools as ft
from woodwork.logical_types import Categorical

warnings.filterwarnings("ignore")
np.random.seed(42)

## 1. Создание связанных таблиц

In [2]:
n_customers, n_products, n_orders, n_items = 50, 20, 300, 1000

customers = pd.DataFrame({
    "client_id": range(1, n_customers + 1),
    "name": [f"Клиент_{i}" for i in range(1, n_customers + 1)],
    "join_date": pd.to_datetime("2023-01-01")
                 + pd.to_timedelta(np.random.randint(0, 365, n_customers), unit="D"),
    "region": np.random.choice(["Москва", "СПб", "Казань", "Новосибирск", "Екатеринбург"], n_customers),
})

products = pd.DataFrame({
    "product_id": range(1, n_products + 1),
    "category": np.random.choice(["Электроника", "Одежда", "Книги", "Дом"], n_products),
    "price": np.round(np.random.uniform(100, 5000, n_products), 2),
})

order_clients = np.random.choice(customers["client_id"], n_orders)
join = customers.set_index("client_id").loc[order_clients, "join_date"].values
orders = pd.DataFrame({
    "order_id": range(1, n_orders + 1),
    "client_id": order_clients,
    "order_date": pd.to_datetime(join) + pd.to_timedelta(np.random.randint(1, 365, n_orders), unit="D"),
})

order_items = pd.DataFrame({
    "item_id": range(1, n_items + 1),
    "order_id": np.random.choice(orders["order_id"], n_items),
    "product_id": np.random.choice(products["product_id"], n_items),
    "quantity": np.random.randint(1, 6, n_items),
})

amount = (order_items.merge(products, on="product_id")
          .assign(s=lambda d: d["price"] * d["quantity"])
          .groupby("order_id")["s"].sum())
orders["total_amount"] = orders["order_id"].map(amount).fillna(0).round(2)

for name, df in [("customers", customers), ("orders", orders),
                 ("products", products), ("order_items", order_items)]:
    print(name, df.shape)

customers (50, 4)
orders (300, 4)
products (20, 3)
order_items (1000, 4)


In [3]:
orders.head()

,order_id,client_id,order_date,total_amount
0,1,39,2023-05-27,23850.80
1,2,45,2023-03-20,14435.88
2,3,15,2024-05-12,31193.26
3,4,43,2024-06-19,42406.36
4,5,29,2024-04-06,36736.83


## 2. EntitySet и связи

In [5]:
es = ft.EntitySet(id="shop")

es = es.add_dataframe(dataframe_name="customers", dataframe=customers,
                      index="client_id", time_index="join_date",
                      logical_types={"region": Categorical})
es = es.add_dataframe(dataframe_name="orders", dataframe=orders,
                      index="order_id", time_index="order_date")
es = es.add_dataframe(dataframe_name="products", dataframe=products,
                      index="product_id", logical_types={"category": Categorical})
es = es.add_dataframe(dataframe_name="order_items", dataframe=order_items,
                      index="item_id")

es = es.add_relationship("customers", "client_id", "orders", "client_id")
es = es.add_relationship("orders", "order_id", "order_items", "order_id")
es = es.add_relationship("products", "product_id", "order_items", "product_id")

es

Entityset: shop
  DataFrames:
    customers [Rows: 50, Columns: 4]
    orders [Rows: 300, Columns: 4]
    products [Rows: 20, Columns: 3]
    order_items [Rows: 1000, Columns: 4]
  Relationships:
    orders.client_id -> customers.client_id
    order_items.order_id -> orders.order_id
    order_items.product_id -> products.product_id

## 3. Генерация признаков (DFS, max_depth=2)

In [6]:
feature_matrix, feature_defs = ft.dfs(
    entityset=es,
    target_dataframe_name="customers",
    max_depth=2,
    agg_primitives=["sum", "mean", "count", "max", "min", "std", "num_unique", "mode"],
    trans_primitives=["year", "month", "weekday", "day"],
)

## 4. Результат

In [7]:
print("Размер матрицы признаков:", feature_matrix.shape)
print("Количество признаков:", len(feature_defs))

Размер матрицы признаков: (50, 55)
Количество признаков: 55


In [8]:
for i, f in enumerate(feature_defs, 1):
    print(i, f.get_name())

1 region
2 COUNT(orders)
3 MAX(orders.total_amount)
4 MEAN(orders.total_amount)
5 MIN(orders.total_amount)
6 STD(orders.total_amount)
7 SUM(orders.total_amount)
8 COUNT(order_items)
9 MAX(order_items.quantity)
10 MEAN(order_items.quantity)
11 MIN(order_items.quantity)
12 STD(order_items.quantity)
13 SUM(order_items.quantity)
14 DAY(join_date)
15 MONTH(join_date)
16 WEEKDAY(join_date)
17 YEAR(join_date)
18 MAX(orders.COUNT(order_items))
19 MAX(orders.MEAN(order_items.quantity))
20 MAX(orders.MIN(order_items.quantity))
21 MAX(orders.STD(order_items.quantity))
22 MAX(orders.SUM(order_items.quantity))
23 MEAN(orders.COUNT(order_items))
24 MEAN(orders.MAX(order_items.quantity))
25 MEAN(orders.MEAN(order_items.quantity))
26 MEAN(orders.MIN(order_items.quantity))
27 MEAN(orders.STD(order_items.quantity))
28 MEAN(orders.SUM(order_items.quantity))
29 MIN(orders.COUNT(order_items))
30 MIN(orders.MAX(order_items.quantity))
31 MIN(orders.MEAN(order_items.quantity))
32 MIN(orders.STD(order_items.qu

In [9]:
feature_matrix.head()

,region,COUNT(orders),MAX(orders.total_amount),MEAN(orders.total_amount),MIN(orders.total_amount),STD(orders.total_amount),SUM(orders.total_amount),COUNT(order_items),MAX(order_items.quantity),MEAN(order_items.quantity),...,STD(orders.SUM(order_items.quantity)),SUM(orders.MAX(order_items.quantity)),SUM(orders.MEAN(order_items.quantity)),SUM(orders.MIN(order_items.quantity)),SUM(orders.STD(order_items.quantity)),MAX(order_items.orders.total_amount),MEAN(order_items.orders.total_amount),MIN(order_items.orders.total_amount),STD(order_items.orders.total_amount),SUM(order_items.orders.total_amount)
client_id,,,,,,,,,,,,,,,,,,,,,
7,Новосибирск,5,23165.88,14907.276000,6758.25,8037.183747,74536.38,13,5.0,2.846154,...,3.507136,21.0,16.466667,13.0,3.918326,23165.88,18578.072308,6758.25,6121.661850,241514.94
45,Москва,5,63922.39,26287.890000,14435.88,21172.022537,131439.45,19,5.0,2.684211,...,7.362065,19.0,13.208333,7.0,6.295802,63922.39,36808.128421,14435.88,23829.053056,699354.44
26,Новосибирск,5,49403.51,26468.826000,9146.51,15226.756905,132344.13,21,5.0,2.761905,...,6.503845,22.0,13.559524,6.0,7.585797,49403.51,31230.985714,9146.51,14577.536097,655850.70
30,Екатеринбург,7,47474.46,25799.745714,1982.78,18782.860267,180598.22,25,5.0,3.000000,...,7.250616,29.0,20.285714,11.0,9.968871,47474.46,33400.540000,1982.78,16013.229811,835013.50
37,Новосибирск,6,43357.94,24329.005000,9997.39,13351.566784,145974.03,16,5.0,3.125000,...,3.265986,25.0,19.250000,14.0,6.802362,43357.94,25597.622500,9997.39,12635.090125,409561.96
